# AutoGen Lecture Notes: GroupChat & Orchestration

**Topic:** GroupChat & Orchestration in Microsoft AutoGen  
**Level:** Beginner to Intermediate  
**Goal:** Understand how multiple AI agents collaborate using AutoGen teams such as `RoundRobinGroupChat`, `SelectorGroupChat`, termination conditions, orchestration strategies, and OpenAI-backed LLM calls.

> These notes use the current AutoGen AgentChat style:
> `autogen-agentchat` + `autogen-ext[openai]`.


## 1. Learning Objectives

By the end of this notebook, you should be able to:

1. Explain what **GroupChat** means in AutoGen.
2. Understand the difference between a single-agent flow and a multi-agent flow.
3. Create multiple `AssistantAgent` agents with different responsibilities.
4. Use OpenAI LLM calls through `OpenAIChatCompletionClient`.
5. Orchestrate agents using:
   - `RoundRobinGroupChat`
   - `SelectorGroupChat`
6. Add termination conditions to avoid infinite conversations.
7. Build a practical QA/SDET-style multi-agent workflow.
8. Understand where GroupChat fits in real-world GenAI applications.


## 2. What is GroupChat in AutoGen?

In AutoGen, **GroupChat** is a multi-agent collaboration pattern.

Instead of one agent answering everything, you create several agents, where each agent has a specialized role.

Example:

| Agent | Responsibility |
|---|---|
| Requirement Analyst | Understands the user story |
| Test Designer | Creates test scenarios |
| Automation Engineer | Writes automation approach/code |
| Reviewer | Reviews and improves the output |

All agents share a common conversation thread. Each agent reads previous messages and contributes based on its role.

AutoGen documentation describes group chat as a design pattern where agents share a common thread of messages, and each participant is specialized for a particular task.


## 3. Why GroupChat is Useful

A single LLM agent can do many things, but it can also mix responsibilities.

GroupChat helps separate responsibilities:

```text
Single Agent:
User → One AI Agent → Final Answer

GroupChat:
User → Analyst Agent → Designer Agent → Developer Agent → Reviewer Agent → Final Output
```

Benefits:

1. **Role specialization**  
   Each agent focuses on one responsibility.

2. **Better quality**  
   A reviewer agent can catch gaps.

3. **Structured reasoning**  
   Complex tasks can be decomposed into smaller tasks.

4. **Reusable workflow**  
   Same team can be reused for different tasks.

5. **Closer to real teams**  
   Similar to BA → QA → Developer → Reviewer collaboration.


## 4. Installation

Run this cell once in your environment.

> Note: You need a valid OpenAI API key for the live LLM examples.


In [6]:
# Install AutoGen AgentChat and OpenAI extension
# Run this in a notebook cell or terminal.

%pip install -U autogen-agentchat autogen-ext[openai] python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 5. Environment Setup for OpenAI API Key

Create a `.env` file in the same folder as this notebook:

```text
OPENAI_API_KEY=your_openai_api_key_here
```

Do **not** hardcode the key inside the notebook.


In [8]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY:
    print("OPENAI_API_KEY loaded successfully.")
else:
    print("OPENAI_API_KEY not found. Add it to a .env file before running live LLM examples.")

OPENAI_API_KEY loaded successfully.


## 6. Core AutoGen Concepts

### 6.1 Agent

An agent is an AI worker with:

- name
- role/instructions
- model client
- optional tools
- memory/context behavior

In AutoGen AgentChat, `AssistantAgent` is commonly used for LLM-backed agents.

### 6.2 Model Client

The model client connects AutoGen to an LLM provider.

For OpenAI, we use:

```python
OpenAIChatCompletionClient
```

### 6.3 Team

A team is a group of agents working together.

Common team types:

| Team Type | Meaning |
|---|---|
| `RoundRobinGroupChat` | Agents speak one after another in fixed order |
| `SelectorGroupChat` | LLM selects which agent should speak next |
| `Swarm` | Agents hand off control to each other |

### 6.4 Termination Condition

A termination condition stops the conversation.

Without termination, a group chat can continue indefinitely.


## 7. Create an OpenAI Model Client

The model client is shared by the agents.

You can use a small and cost-effective model for learning, such as `gpt-4o-mini`, or another OpenAI chat model available to your account.


In [11]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY
)

print("OpenAI model client created.")

OpenAI model client created.


## 8. Example 1: Single Agent Baseline

Before GroupChat, let us first create one simple agent.


In [13]:
import asyncio
from autogen_agentchat.agents import AssistantAgent

single_agent = AssistantAgent(
    name="qa_assistant",
    model_client=model_client,
    system_message=(
        "You are a senior QA automation engineer. "
        "Explain concepts clearly and practically."
    )
)

async def run_single_agent():
    result = await single_agent.run(
        task="Explain why Playwright is useful for testing a modern ecommerce web application."
    )
    return result

# In Jupyter, use: result = await run_single_agent()
# If running as a script, use: asyncio.run(run_single_agent())

result = await run_single_agent()
print(result.messages[-1].content)

Playwright is an open-source automation library developed by Microsoft that is particularly useful for end-to-end (E2E) testing of modern web applications, including eCommerce platforms. Here are several key reasons why Playwright is beneficial for testing eCommerce web applications:

### 1. **Cross-Browser Testing**
Playwright supports multiple browsers — Chromium, Firefox, and WebKit (the engine behind Safari) — allowing you to test your eCommerce application across all major browsers. This is crucial for eCommerce sites, as different users may have different browsers, and maintaining a consistent experience across them is essential.

### 2. **Multi-Device Testing**
With Playwright, you can simulate devices and viewports, which is critical in eCommerce due to the variety of devices consumers use to shop (mobile, tablet, desktop). This allows you to test how your application responds and performs on different screen sizes and orientations.

### 3. **Robust Automation Features**
Playwr

## 9. Limitations of a Single Agent

A single agent can answer the question, but it may try to do everything:

- understand requirements
- design tests
- write automation
- review quality

For complex work, this is not ideal.

That is where GroupChat helps.


## 10. Example 2: RoundRobinGroupChat

`RoundRobinGroupChat` is the simplest orchestration pattern.

Agents speak in a fixed order:

```text
Agent 1 → Agent 2 → Agent 3 → Agent 4 → repeat until stopped
```

This is useful when the workflow is predictable.


In [16]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination

requirement_analyst = AssistantAgent(
    name="requirement_analyst",
    model_client=model_client,
    system_message=(
        "You are a business analyst. "
        "Extract clear functional requirements from the user's request. "
        "Keep your response concise."
    )
)

test_designer = AssistantAgent(
    name="test_designer",
    model_client=model_client,
    system_message=(
        "You are a senior QA test designer. "
        "Create practical test scenarios and edge cases from requirements."
    )
)

automation_engineer = AssistantAgent(
    name="automation_engineer",
    model_client=model_client,
    system_message=(
        "You are a Playwright TypeScript automation engineer. "
        "Suggest automation approach, page objects, and stable locator strategy."
    )
)

reviewer = AssistantAgent(
    name="reviewer",
    model_client=model_client,
    system_message=(
        "You are a QA architect. "
        "Review the previous agents' output and provide final recommendations. "
        "End your final answer with TERMINATE."
    )
)

termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(max_messages=8)

qa_team = RoundRobinGroupChat(
    participants=[
        requirement_analyst,
        test_designer,
        automation_engineer,
        reviewer
    ],
    termination_condition=termination
)

async def run_round_robin_groupchat():
    task = (
        "We have an ecommerce saree website. "
        "User should search for a saree, open product details, add it to cart, "
        "and proceed to checkout. Create a QA test strategy."
    )
    result = await qa_team.run(task=task)
    return result

round_robin_result = await run_round_robin_groupchat()

for message in round_robin_result.messages:
    source = getattr(message, "source", "unknown")
    content = getattr(message, "content", "")
    print(f"\n--- {source} ---\n{content}")


--- user ---
We have an ecommerce saree website. User should search for a saree, open product details, add it to cart, and proceed to checkout. Create a QA test strategy.

--- requirement_analyst ---
Functional Requirements:

1. **Search Functionality**:
   - Users should be able to search for sarees using keywords.
   - The search results should display relevant sarees based on the search query.

2. **Product Details Access**:
   - Users should be able to click on a saree to view detailed product information, including images, descriptions, prices, and available sizes/colors.

3. **Add to Cart Feature**:
   - Users should have the option to select a size/color and add the saree to their shopping cart.
   - The cart should update to reflect the addition of the saree.

4. **Checkout Process**:
   - Users should be able to navigate to the cart from any page on the website.
   - The checkout process should incorporate customer details entry, payment processing, and order confirmation.

5

## 11. Understanding the RoundRobin Output

In the previous example:

1. The **Requirement Analyst** clarified the business flow.
2. The **Test Designer** created scenarios.
3. The **Automation Engineer** suggested Playwright implementation ideas.
4. The **Reviewer** checked the output and ended with `TERMINATE`.

The orchestration is deterministic because the speaking order is fixed.


## 12. Termination Conditions

Termination conditions are very important.

Without termination, a multi-agent conversation can continue longer than expected and increase cost.

Common patterns:

| Termination | Purpose |
|---|---|
| `MaxMessageTermination` | Stop after fixed number of messages |
| `TextMentionTermination` | Stop when a keyword appears |
| Combined conditions | Stop based on either rule |

Example:

```python
termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(max_messages=8)
```

This means:

```text
Stop if someone says TERMINATE
OR
Stop after 8 messages
```


## 13. Example 3: SelectorGroupChat

`SelectorGroupChat` is more dynamic.

Instead of fixed turn order, an LLM decides who should speak next.

This is useful when the conversation is not predictable.

Example:

```text
User task
↓
LLM selector decides:
Requirement Analyst should speak first
↓
Then Test Designer
↓
Then Reviewer
```

This is closer to intelligent orchestration.


In [20]:
from autogen_agentchat.teams import SelectorGroupChat

selector_team = SelectorGroupChat(
    participants=[
        requirement_analyst,
        test_designer,
        automation_engineer,
        reviewer
    ],
    model_client=model_client,
    termination_condition=termination,
    allow_repeated_speaker=False
)

async def run_selector_groupchat():
    task = (
        "Create an end-to-end testing plan for login, product search, add to cart, "
        "wishlist, and checkout in a saree ecommerce application. "
        "Include which tests should be smoke and which should be regression."
    )
    result = await selector_team.run(task=task)
    return result

selector_result = await run_selector_groupchat()

for message in selector_result.messages:
    source = getattr(message, "source", "unknown")
    content = getattr(message, "content", "")
    print(f"\n--- {source} ---\n{content}")


--- user ---
Create an end-to-end testing plan for login, product search, add to cart, wishlist, and checkout in a saree ecommerce application. Include which tests should be smoke and which should be regression.

--- test_designer ---
### End-to-End Testing Plan for E-commerce Saree Application

#### Objective
To validate the integrated functionality of key features in the saree e-commerce application to ensure that they work together seamlessly and provide a pleasant user experience. The key features to be tested include login, product search, add to cart, wishlist, and checkout processes.

---

### Test Scope

#### Features to be Tested
1. User Login
2. Product Search
3. Add to Cart
4. Wishlist Functionality
5. Checkout Process

---

### Testing Types

1. **Smoke Testing**: Quick checks to ensure that the essential functionalities are working before deeper testing is conducted. 
2. **Regression Testing**: Comprehensive testing after any code change to ensure that existing functional

## 14. RoundRobin vs SelectorGroupChat

| Feature | RoundRobinGroupChat | SelectorGroupChat |
|---|---|---|
| Speaker order | Fixed | Dynamic |
| Best for | Predictable workflows | Flexible workflows |
| Cost | Usually lower | Slightly higher because selector uses LLM |
| Control | High | Medium |
| Intelligence | Lower | Higher |
| Example | BA → QA → Dev → Reviewer | LLM chooses next best agent |

### Recommendation

For learning and initial projects:

```text
Start with RoundRobinGroupChat
```

For more advanced AI orchestration:

```text
Use SelectorGroupChat
```


## 15. Example 4: QA Automation Framework Design Team

Now let us create a multi-agent team specifically for your use case:

**Building a Playwright automation framework for an ecommerce application.**


In [23]:
framework_architect = AssistantAgent(
    name="framework_architect",
    model_client=model_client,
    system_message=(
        "You are a test automation architect. "
        "Design scalable Playwright TypeScript framework architecture. "
        "Focus on folder structure, fixtures, configs, page objects, and CI/CD."
    )
)

playwright_engineer = AssistantAgent(
    name="playwright_engineer",
    model_client=model_client,
    system_message=(
        "You are a senior Playwright TypeScript engineer. "
        "Provide implementation-level guidance and example code when needed."
    )
)

api_test_engineer = AssistantAgent(
    name="api_test_engineer",
    model_client=model_client,
    system_message=(
        "You are an API testing specialist. "
        "Design API testing layer for FastAPI backend validation."
    )
)

qa_reviewer = AssistantAgent(
    name="qa_reviewer",
    model_client=model_client,
    system_message=(
        "You are a QA manager and reviewer. "
        "Challenge over-engineering, identify missing areas, and finalize recommendations. "
        "End final response with TERMINATE."
    )
)

framework_team = RoundRobinGroupChat(
    participants=[
        framework_architect,
        playwright_engineer,
        api_test_engineer,
        qa_reviewer
    ],
    termination_condition=TextMentionTermination("TERMINATE") | MaxMessageTermination(max_messages=8)
)

async def design_framework_with_groupchat():
    task = (
        "Design a production-grade Playwright TypeScript automation framework "
        "for a saree ecommerce application with React frontend, FastAPI backend, "
        "PostgreSQL database, product catalog, cart, wishlist, checkout, "
        "admin product management, and GitHub Actions CI/CD."
    )
    result = await framework_team.run(task=task)
    return result

framework_result = await design_framework_with_groupchat()

for message in framework_result.messages:
    source = getattr(message, "source", "unknown")
    content = getattr(message, "content", "")
    print(f"\n--- {source} ---\n{content}")


--- user ---
Design a production-grade Playwright TypeScript automation framework for a saree ecommerce application with React frontend, FastAPI backend, PostgreSQL database, product catalog, cart, wishlist, checkout, admin product management, and GitHub Actions CI/CD.

--- framework_architect ---
Designing a production-grade Playwright TypeScript automation framework for a saree eCommerce application entails creating a well-organized structure to manage tests, Page Objects, configurations, fixtures, and CI/CD. Here’s an architecture outline for your framework, along with explanations of each component.

### Folder Structure

```
saree-ecommerce-tests/
│
├── .github/
│   └── workflows/
│       └── ci.yml
│
├── src/
│   ├── config/
│   │   ├── globalConfig.ts
│   │   ├── playwright.config.ts
│   │   └── testConfig.ts
│   │
│   ├── fixtures/
│   │   ├── authFixture.ts
│   │   └── dataFixture.ts
│   │
│   ├── pageObjects/
│   │   ├── HomePage.ts
│   │   ├── ProductPage.ts
│   │   ├── Car

## 16. Orchestration Design Patterns

AutoGen orchestration is about controlling:

1. **Who speaks next**
2. **When the conversation stops**
3. **What each agent is allowed to do**
4. **How messages flow**
5. **Whether humans are involved**

Common orchestration styles:

### 16.1 Sequential Orchestration

Fixed order.

```text
Planner → Executor → Reviewer
```

Good for predictable tasks.

### 16.2 Selector-Based Orchestration

An LLM selects the next speaker.

```text
User task → Selector decides next agent
```

Good for dynamic tasks.

### 16.3 Human-in-the-Loop Orchestration

A human approves or corrects before continuing.

Good for high-risk tasks.

### 16.4 Nested Team Orchestration

A group chat can be part of a larger group chat.

Example:

```text
Main Team
├── QA Team
├── Dev Team
└── Security Team
```

Useful for enterprise workflows.


## 17. Practical Tips for GroupChat Design

### 17.1 Keep Agent Roles Clear

Bad role:

```text
You are a helpful assistant.
```

Better role:

```text
You are a senior Playwright automation engineer. 
Focus only on automation design and code-level recommendations.
```

### 17.2 Avoid Too Many Agents Initially

Start with 3 or 4 agents.

Too many agents can increase cost and produce repetitive output.

### 17.3 Always Add Termination Conditions

Use both:

```python
TextMentionTermination("TERMINATE")
MaxMessageTermination(max_messages=8)
```

### 17.4 Use RoundRobin First

RoundRobin is easier to debug.

Move to SelectorGroupChat after your workflow is stable.

### 17.5 Design Agents Like Real Team Members

Example for QA:

```text
Business Analyst
Test Designer
Automation Engineer
Reviewer
```


## 18. Example 5: Generate Test Scenarios Using Multi-Agent QA Team

This example asks the team to produce concrete test scenarios.


In [27]:
scenario_team = RoundRobinGroupChat(
    participants=[
        requirement_analyst,
        test_designer,
        reviewer
    ],
    termination_condition=TextMentionTermination("TERMINATE") | MaxMessageTermination(max_messages=6)
)

async def generate_ecommerce_test_scenarios():
    task = (
        "Generate test scenarios for an ecommerce saree website. "
        "Features: product listing, category filter, product details, add to cart, "
        "wishlist, checkout, and order confirmation. "
        "Separate the scenarios into smoke, regression, and negative test cases."
    )
    result = await scenario_team.run(task=task)
    return result

scenario_result = await generate_ecommerce_test_scenarios()

for message in scenario_result.messages:
    source = getattr(message, "source", "unknown")
    content = getattr(message, "content", "")
    print(f"\n--- {source} ---\n{content}")


--- user ---
Generate test scenarios for an ecommerce saree website. Features: product listing, category filter, product details, add to cart, wishlist, checkout, and order confirmation. Separate the scenarios into smoke, regression, and negative test cases.

--- requirement_analyst ---
### Test Scenarios for Ecommerce Saree Website

#### Smoke Test Cases
1. **Product Listing**:
   - Verify that the homepage displays a list of sarees.
  
2. **Category Filter**:
   - Check that users can filter sarees by category (e.g., casual, party wear).

3. **Product Details**:
   - Ensure users can open product details from the listing page.

4. **Add to Cart**:
   - Validate that users can successfully add a product to the cart.

5. **Checkout**:
   - Confirm users can proceed to checkout from the cart.

6. **Order Confirmation**:
   - Check that users receive an order confirmation page after purchase.

#### Regression Test Cases
1. **Product Listing**:
   - Verify that all sarees are displayed c

## 19. Example 6: Streaming GroupChat Output

For better user experience, you can stream messages as they are generated.

Streaming is useful in applications where you want to show agent collaboration live.


In [29]:
from autogen_agentchat.ui import Console

stream_team = RoundRobinGroupChat(
    participants=[
        requirement_analyst,
        test_designer,
        reviewer
    ],
    termination_condition=TextMentionTermination("TERMINATE") | MaxMessageTermination(max_messages=6)
)

async def stream_groupchat():
    task = (
        "Create a concise QA plan for checkout functionality in an ecommerce app. "
        "Include functional, negative, and automation considerations."
    )
    await Console(stream_team.run_stream(task=task))

await stream_groupchat()

---------- TextMessage (user) ----------
Create a concise QA plan for checkout functionality in an ecommerce app. Include functional, negative, and automation considerations.
---------- TextMessage (requirement_analyst) ----------
### QA Plan for Checkout Functionality in an Ecommerce App

#### 1. **Objective**
To ensure the checkout process is functional, seamless, and error-free for users, providing a secure transaction experience.

#### 2. **Scope**
Testing will cover all aspects of the checkout functionality, including cart access, user input, payment processing, and order confirmation.

#### 3. **Functional Test Cases**
- **Cart Access**: 
  - Verify users can view the cart from any page.
  
- **User Information Entry**: 
  - Ensure all fields (name, address, contact, payment details) accept valid inputs.
  
- **Payment Processing**: 
  - Test various payment methods (credit card, PayPal, etc.) for processing during checkout.
  
- **Order Summary**: 
  - Validate that the order su

## 20. Common Mistakes

### Mistake 1: No termination condition

This can cause long-running conversations.

### Mistake 2: Vague agent roles

Vague agents produce overlapping answers.

### Mistake 3: Too many agents

More agents does not always mean better output.

### Mistake 4: Using SelectorGroupChat too early

Start with RoundRobin to validate the workflow first.

### Mistake 5: Not controlling cost

Every agent response uses LLM tokens. Use smaller models for learning.


## 21. Real-World Use Cases

### 21.1 QA Automation

Agents:

- Requirement Analyst
- Test Designer
- Automation Engineer
- Reviewer

Output:

- test scenarios
- Playwright framework design
- locator strategy
- CI/CD strategy

### 21.2 Code Review

Agents:

- Code Reviewer
- Security Reviewer
- Performance Reviewer
- Refactoring Advisor

### 21.3 Product Requirement Analysis

Agents:

- Product Manager
- Business Analyst
- UX Reviewer
- QA Lead

### 21.4 DevSecOps Pipeline Design

Agents:

- DevOps Engineer
- Security Engineer
- QA Engineer
- Reviewer


## 22. Mini Project: AutoGen QA Team for Ecommerce Testing

### Problem Statement

Build a multi-agent team that can analyze an ecommerce feature and produce:

1. Functional test scenarios
2. Negative test scenarios
3. Automation approach
4. Page Object recommendations
5. API test recommendations
6. Final review

### Suggested Agents

```text
requirement_analyst
test_designer
automation_engineer
api_test_engineer
reviewer
```

### Suggested Orchestration

Use `RoundRobinGroupChat` first.

Then try `SelectorGroupChat`.


In [33]:
mini_project_team = RoundRobinGroupChat(
    participants=[
        requirement_analyst,
        test_designer,
        automation_engineer,
        api_test_engineer,
        reviewer
    ],
    termination_condition=TextMentionTermination("TERMINATE") | MaxMessageTermination(max_messages=10)
)

async def mini_project():
    task = '''
    Feature: Admin Product Creation

    Admin should be able to create a saree product with:
    - name
    - description
    - price
    - MRP
    - discount
    - category
    - fabric
    - color
    - blouse information
    - stock by size
    - product images

    Generate complete QA testing output:
    1. Functional test cases
    2. Negative test cases
    3. API tests
    4. Playwright Page Object suggestions
    5. Test data strategy
    6. Final review
    '''
    result = await mini_project_team.run(task=task)
    return result

mini_project_result = await mini_project()

for message in mini_project_result.messages:
    source = getattr(message, "source", "unknown")
    content = getattr(message, "content", "")
    print(f"\n--- {source} ---\n{content}")


--- user ---

    Feature: Admin Product Creation

    Admin should be able to create a saree product with:
    - name
    - description
    - price
    - MRP
    - discount
    - category
    - fabric
    - color
    - blouse information
    - stock by size
    - product images

    Generate complete QA testing output:
    1. Functional test cases
    2. Negative test cases
    3. API tests
    4. Playwright Page Object suggestions
    5. Test data strategy
    6. Final review
    

--- requirement_analyst ---
### QA Testing Output for Admin Product Creation Feature

#### 1. Functional Test Cases
1. **Create Product with Valid Details**:
   - Verify that an admin can successfully create a saree product with all required fields (name, description, price, etc.).
  
2. **Validation of Required Fields**:
   - Ensure that all mandatory fields must be filled before submission, and the system prompts errors for any missing information.
   
3. **Price Validation**:
   - Test that the price a

In [34]:
mini_project()

<coroutine object mini_project at 0x00000287B441DF20>

## 23. Clean Up Model Client

Some AutoGen model clients expose a `close()` method. Use it after finishing live examples.


In [36]:
await model_client.close()
print("Model client closed.")

Model client closed.


## 24. Interview-Style Questions

### Q1. What is GroupChat in AutoGen?

GroupChat is a multi-agent pattern where multiple specialized agents collaborate in a shared conversation thread.

### Q2. What is orchestration?

Orchestration is the control mechanism that decides how agents interact, who speaks next, what role each agent plays, and when the workflow stops.

### Q3. Difference between RoundRobinGroupChat and SelectorGroupChat?

RoundRobin uses fixed speaker order. SelectorGroupChat uses an LLM to dynamically select the next speaker.

### Q4. Why are termination conditions important?

They prevent infinite loops, reduce cost, and make the conversation predictable.

### Q5. How would you use AutoGen GroupChat in testing?

You can create agents such as Requirement Analyst, Test Designer, Automation Engineer, API Tester, and Reviewer to generate test strategies, test cases, automation designs, and review outputs.


## 25. Summary

In this notebook, you learned:

- What GroupChat means in AutoGen
- How multiple agents collaborate
- How to use OpenAI through `OpenAIChatCompletionClient`
- How to create `AssistantAgent`
- How to orchestrate agents using:
  - `RoundRobinGroupChat`
  - `SelectorGroupChat`
- How to add termination conditions
- How to apply GroupChat to QA automation and ecommerce testing

### Recommended Next Topics

1. AutoGen tools and function calling
2. Human-in-the-loop workflows
3. Agent memory and state
4. Nested teams
5. AutoGen + Playwright integration
6. Building AI QA agents for test case generation


## 26. References

- AutoGen AgentChat teams API documentation  
- AutoGen Group Chat design pattern documentation  
- Microsoft Agent Framework migration guide for AutoGen context and orchestration evolution
